In [19]:
!pip install pandas
!pip install sentence-transformers
!pip install scikit-learn

In [20]:
import pandas as pd
import ast
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [21]:
movies = pd.read_csv("tmdb_5000_movies.csv")

movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [22]:
movies = movies[['title','overview','genres','keywords','vote_average']]

movies = movies.dropna()

movies.head()

,title,overview,genres,keywords,vote_average
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",7.2
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",6.9
2,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",6.3
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",7.6
4,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",6.1


In [23]:
import ast

def clean_json(text):
    try:
        items = ast.literal_eval(text)
        return " ".join([i["name"] for i in items])
    except:
        return ""

movies["genres"] = movies["genres"].apply(clean_json)
movies["keywords"] = movies["keywords"].apply(clean_json)

movies.head()

,title,overview,genres,keywords,vote_average
0,Avatar,"In the 22nd century, a paraplegic Marine is di...",Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,7.2
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,6.9
2,Spectre,A cryptic message from Bond’s past sends him o...,Action Adventure Crime,spy based on novel secret agent sequel mi6 bri...,6.3
3,The Dark Knight Rises,Following the death of District Attorney Harve...,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,7.6
4,John Carter,"John Carter is a war-weary, former military ca...",Action Adventure Science Fiction,based on novel mars medallion space travel pri...,6.1


In [24]:
movies["text"] = movies["overview"] + " " + movies["genres"] + " " + movies["keywords"]

movies.head()

,title,overview,genres,keywords,vote_average,text
0,Avatar,"In the 22nd century, a paraplegic Marine is di...",Action Adventure Fantasy Science Fiction,culture clash future space war space colony so...,7.2,"In the 22nd century, a paraplegic Marine is di..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",Adventure Fantasy Action,ocean drug abuse exotic island east india trad...,6.9,"Captain Barbossa, long believed to be dead, ha..."
2,Spectre,A cryptic message from Bond’s past sends him o...,Action Adventure Crime,spy based on novel secret agent sequel mi6 bri...,6.3,A cryptic message from Bond’s past sends him o...
3,The Dark Knight Rises,Following the death of District Attorney Harve...,Action Crime Drama Thriller,dc comics crime fighter terrorist secret ident...,7.6,Following the death of District Attorney Harve...
4,John Carter,"John Carter is a war-weary, former military ca...",Action Adventure Science Fiction,based on novel mars medallion space travel pri...,6.1,"John Carter is a war-weary, former military ca..."


In [25]:
from sentence_transformers import SentenceTransformer

In [8]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(movies["text"].tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/150 [00:00<?, ?it/s]

In [9]:
similarity_matrix = cosine_similarity(embeddings)

In [10]:
def recommend(movie_title, top_n=5):

    if movie_title not in movies["title"].values:
        return "Movie not found."

    idx = movies[movies["title"] == movie_title].index[0]

    scores = list(enumerate(similarity_matrix[idx]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    scores = scores[1:top_n+1]

    movie_indices = [i[0] for i in scores]

    return movies["title"].iloc[movie_indices]

In [11]:
recommend("Avatar")

1294                  Serenity
2403                    Aliens
1352                   Gattaca
56            Star Trek Beyond
4187    Independence Daysaster
Name: title, dtype: object

In [12]:
def recommend_with_explanation(movie_title, top_n=5):

    if movie_title not in movies["title"].values:
        return "Movie not found."

    idx = movies[movies["title"] == movie_title].index[0]

    scores = list(enumerate(similarity_matrix[idx]))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    results = []

    for i in scores[1:top_n+1]:

        movie = movies.iloc[i[0]]

        explanation = f"Recommended because it shares similar genres ({movie['genres']}) and themes with {movie_title}."

        results.append({
            "title": movie["title"],
            "explanation": explanation
        })

    return results

In [29]:
def recommend_by_genre(genre, top_n=5):

    filtered = movies[movies["genres"].str.contains(genre, case=False, na=False)]

    if len(filtered) == 0:
        return "No movies found for this genre."

    results = []

    for _, movie in filtered.head(top_n).iterrows():

        explanation = f"Recommended because it belongs to the {genre} genre."

        results.append({
            "title": movie["title"],
            "explanation": explanation
        })

    return results

In [26]:
def show_recommendations(movie):
    results = recommend_with_explanation(movie)

    for r in results:
        print("Movie:", r["title"])
        print("Explanation:", r["explanation"])
        print()

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

In [28]:
recommend_with_explanation("Avatar")

[{'title': 'Serenity',
  'explanation': 'Recommended because it shares similar genres (Science Fiction Action Adventure Thriller) and themes with Avatar.'},
 {'title': 'Aliens',
  'explanation': 'Recommended because it shares similar genres (Horror Action Thriller Science Fiction) and themes with Avatar.'},
 {'title': 'Gattaca',
  'explanation': 'Recommended because it shares similar genres (Thriller Science Fiction Mystery Romance) and themes with Avatar.'},
 {'title': 'Star Trek Beyond',
  'explanation': 'Recommended because it shares similar genres (Action Adventure Science Fiction) and themes with Avatar.'},
 {'title': 'Independence Daysaster',
  'explanation': 'Recommended because it shares similar genres (Action Science Fiction) and themes with Avatar.'}]

In [30]:
recommend_by_genre("Science Fiction")

[{'title': 'Avatar',
  'explanation': 'Recommended because it belongs to the Science Fiction genre.'},
 {'title': 'John Carter',
  'explanation': 'Recommended because it belongs to the Science Fiction genre.'},
 {'title': 'Avengers: Age of Ultron',
  'explanation': 'Recommended because it belongs to the Science Fiction genre.'},
 {'title': 'Superman Returns',
  'explanation': 'Recommended because it belongs to the Science Fiction genre.'},
 {'title': 'Man of Steel',
  'explanation': 'Recommended because it belongs to the Science Fiction genre.'}]

In [31]:
import numpy as np

# Precision@K
def precision_at_k(recommended, relevant, k=5):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k


# Recall@K
def recall_at_k(recommended, relevant, k=5):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant)


# Diversity Score
def diversity_score(recommended):
    genres = movies[movies["title"].isin(recommended)]["genres"]
    
    unique_genres = set()
    
    for g in genres:
        for item in g.split():
            unique_genres.add(item)
            
    return len(unique_genres) / len(recommended)

In [32]:
recommended = [r["title"] for r in recommend_with_explanation("Avatar")]

relevant = ["Aliens", "Serenity", "Gattaca"]

print("Precision@5:", precision_at_k(recommended, relevant))
print("Recall@5:", recall_at_k(recommended, relevant))
print("Diversity:", diversity_score(recommended))

Precision@5: 0.6
Recall@5: 1.0
Diversity: 1.6
